# 🏥 MedShare: Gold Standard Federated Learning

**Definitive System Verification Notebook**

---


In [1]:
## 🛸 PHASE 1: SYSTEM INITIALIZATION (Setup & Infrastructure)
from google.colab import drive
import os, torch, multiprocessing, psutil, socket, time, subprocess, json, urllib.request, shutil

# 1.1 Drive Sync
print("1.1 Mounting Google Drive...")
os.chdir('/content')
drive.mount('/content/drive', force_remount=True)

POSSIBLE_PATHS = [
    '/content/drive/Othercomputers/My laptop/bxp267',
    '/content/drive/MyDrive/bxp267',
    '/content/drive/MyDrive/coding/bxp267'
]
PROJECT_PATH = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)
if not PROJECT_PATH:
    raise Exception("❌ Project folder not found. Please mount Drive and check path.")
%cd "$PROJECT_PATH"

# 1.2 Hardware Pre-flight
print("\n1.2 Calibrating Hardware...")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB VRAM)")
    print("[Hardware] Mode: Colab MAX Performance Enabled")
else: print("⚠️ GPU NOT FOUND. Using CPU fallback.")

# 1.3 Dependency Installation
print("\n1.3 Installing Core Engines...")
!pip install -q "protobuf<6" "cryptography<44" "flwr[simulation]" opacus ucimlrepo web3 kagglehub imbalanced-learn
!npm install -g ganache --legacy-peer-deps 2>&1 | tail -1

# 1.4 Blockchain Initialization
print("\n1.4 Provisioning Blockchain...")
!lsof -ti:8545 | xargs kill -9 > /dev/null 2>&1
subprocess.Popen(["ganache", "--port", "8545", "--miner.blockGasLimit", "0xffffffffffff"], stdout=subprocess.DEVNULL)
time.sleep(5)
!npx hardhat compile > /dev/null 2>&1
os.makedirs("build", exist_ok=True)
for c in ["MedShareTask", "CommitmentRegistry", "Reputation"]:
    shutil.copy(f"artifacts/contracts/{c}.sol/{c}.json", f"build/{c}.json")
!python scripts/deploy_colab.py

print("\n" + "="*70)
print("🏁 INITIALIZATION SUCCESS: MedShare System is LIVE")
print("="*70)

1.1 Mounting Google Drive...
Mounted at /content/drive
/content/drive/Othercomputers/My laptop/bxp267

1.2 Calibrating Hardware...
⚠️ GPU NOT FOUND. Using CPU fallback.

1.3 Installing Core Engines...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.4/254.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.5/587.5 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 4.3 MB/s eta 0:00:00
   ━━━

In [2]:
## 🔧 PHASE 1.5: SELF-HEALING PATCH (SSL Certificate Fix)
# Directly patches fetch_thyroid() by writing the fixed function into data.py.
# Uses string find/replace to locate the old function — no regex, no escape issues.
import sys

data_py_path = 'medshare/data.py'
with open(data_py_path, 'r', encoding='utf-8') as f:
    content = f.read()

# Find start and end of the old fetch_thyroid function
start_marker = 'def fetch_thyroid():'
end_marker = '\ndef fetch_diabetes_hospitals():'

start_idx = content.find(start_marker)
end_idx = content.find(end_marker)

if start_idx == -1 or end_idx == -1:
    print('⚠️  Could not locate fetch_thyroid block — skipping patch.')
else:
    # Build the replacement using a list of lines to avoid any escape issues
    lines = [
        'def fetch_thyroid():',
        '    import pandas as pd, ssl, urllib.request, io',
        '',
        '    cols = [',
        '        "age", "sex", "on_thyroxine", "query_on_thyroxine", "on_antithyroid_medication",',
        '        "sick", "pregnant", "thyroid_surgery", "i131_treatment", "query_hypothyroid",',
        '        "query_hyperthyroid", "lithium", "goitre", "tumor", "hypopituitary", "psych",',
        '        "tsh", "t3", "tt4", "t4u", "fti", "target"',
        '    ]',
        '',
        '    # Method 1: ucimlrepo (preferred, no SSL issues)',
        '    try:',
        '        from ucimlrepo import fetch_ucirepo',
        '        print("[Data] Fetching Thyroid via ucimlrepo...")',
        '        repo = fetch_ucirepo(id=102)',
        '        X = repo.data.features',
        '        y = repo.data.targets',
        '        df = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)',
        '        # Standardize column names to match the definitive schema',
        '        df.columns = cols ',
        '        print(f"[Data] Thyroid loaded via ucimlrepo: {len(df)} rows, {len(df.columns)} cols")',
        '        return df',
        '    except Exception as e:',
        '        print(f"[Data] ucimlrepo failed ({e}), falling back to direct download...")',
        '',
        '    # Method 2: Direct URL with SSL verification bypassed (expired cert fallback)',
        '    base_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/thyroid-disease/"',
        '    ssl_ctx = ssl.create_default_context()',
        '    ssl_ctx.check_hostname = False',
        '    ssl_ctx.verify_mode = ssl.CERT_NONE',
        '',
        '    def read_url(url):',
        '        with urllib.request.urlopen(url, context=ssl_ctx) as resp:',
        '            return pd.read_csv(io.StringIO(resp.read().decode("utf-8")), sep="\\s+", header=None)',
        '',
        '    train_df = read_url(f"{base_url}ann-train.data")',
        '    test_df  = read_url(f"{base_url}ann-test.data")',
        '    df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)',
        '    df.columns = cols',
        '    return df',
    ]
    new_fn = '\n'.join(lines)

    patched = content[:start_idx] + new_fn + '\n' + content[end_idx:]

    with open(data_py_path, 'w', encoding='utf-8') as f:
        f.write(patched)
    print('✅ PATCH APPLIED: fetch_thyroid() now uses ucimlrepo (SSL-safe)')

# Clear any cached medshare imports so the subprocess picks up fresh .py files
for mod in list(sys.modules.keys()):
    if 'medshare' in mod:
        del sys.modules[mod]
print('✅ Module cache cleared')


✅ PATCH APPLIED: fetch_thyroid() now uses ucimlrepo (SSL-safe)
✅ Module cache cleared


In [ ]:
## 🔬 PHASE 2: RIGOROUS SCIENTIFIC SWEEP (System Stress Test)
import os
print("\ud83d\udd25 STARTING GLOBAL STRESS TEST (Admin-Category Sweep)")
print("="*70 + "\n")

experiments = [
    ("Privacy Audit (Audit Sweep)", "mi", "admin_category", 50, 40),
    ("DP Utility Frontier", "dp", "admin_category", 50, 20),
    ("Adversarial Robustness", "robustness", "admin_category", 50, 20),
    ("System Telemetry (Gas/Latency)", "latency", "admin_category", 15, 10)
]

for label, mode, dataset, rounds, epochs in experiments:
    print(f"\n\ud83d\ude80 Phase: {label} ({mode}) on {dataset}...")
    !python federated_survival.py --experiment {mode} --dataset {dataset} --rounds {rounds} --epochs {epochs} --enable_dp True --enable_blockchain True

print("\n" + "="*70)
print("\u2705 SCIENTIFIC SWEEP COMPLETE")
print("="*70)

🔥 STARTING GLOBAL STRESS TEST (Thyroid Disease Sweep)


🚀 Phase: Privacy Audit (Audit Sweep) (mi) on thyroid...
2026-02-22 00:03:34.560542: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-22 00:03:34.567360: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-22 00:03:34.590283: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771718614.633421    1252 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771718614.647060    1252 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771718614.680203

In [ ]:
## 📈 PHASE 3: AUTOMATED ANALYTICS & FINAL VERIFICATION
import os, pandas as pd, matplotlib.pyplot as plt, matplotlib.image as mpimg

# 3.1 Trigger High-Intensity Plotting
print("3.1 Generating Automated Visuals...")
!python test/plot_results.py

# 3.2 Verification Check
print("\n3.2 Final System Integrity Audit...")
csvs = ["exp_mi_results.csv", "exp_dp_results.csv", "exp_robustness_results.csv", "exp_gas_log.csv", "exp_latency_log.csv"]
imgs = ["fig_mi.png", "fig_dp_tradeoff.png", "fig_robustness.png", "fig_gas_costs.png", "fig_latency.png"]

results = []
for f in csvs: results.append(("CSV Data", f, os.path.exists(f"test/{f}") or os.path.exists(f)))
for f in imgs: results.append(("PNG Visual", f, os.path.exists(f"test/{f}") or os.path.exists(f)))

print(pd.DataFrame(results, columns=["Type", "Resource", "Status"]).to_markdown(index=False))

# Check for comparison stats
if os.path.exists("frontend/src/data/comparison_stats.json"):
    with open("frontend/src/data/comparison_stats.json", "r") as f:
        stats = json.load(f)
    print(f"\n🎯 Dataset: {stats.get('dataset_name', 'N/A')}")
    print(f"🎯 Final Fed Accuracy: {stats.get('federated_accuracy', 'N/A')}")
    print(f"🔐 Privacy Budget (ε): {stats.get('security', {}).get('epsilon', 'N/A')}")
    print(f"⛓️ Defense: {stats.get('security', {}).get('defense_type', 'N/A')}")

# 3.3 Display Key Visuals
print("\n3.3 Rendering Definitive Results...")
for p in [f"test/{i}" for i in imgs]:
    if os.path.exists(p):
        print(f"\n🖼️ {p}")
        img = mpimg.imread(p)
        plt.figure(figsize=(10, 5)); plt.imshow(img); plt.axis('off'); plt.show()

print("\n" + "="*70)
csv_count = sum(1 for f in csvs if os.path.exists(f"test/{f}") or os.path.exists(f))
plot_count = sum(1 for f in imgs if os.path.exists(f"test/{f}") or os.path.exists(f))

if csv_count == 5 and plot_count == 5:
    print("✅ FULL EXECUTION COMPLETE - ALL TESTS PASSED")
else:
    print(f"⚠️ PARTIAL EXECUTION - {5-csv_count} data files and {5-plot_count} plots missing")
print("="*70)